In [1]:
#!source nflvenv/bin/activate

In [1]:
import gc
import pandas as pd
import numpy as np
from scipy.spatial import Voronoi, ConvexHull
from shapely.geometry import Polygon, Point
from shapely.errors import GEOSException

In [3]:
df = pd.read_parquet("df_training_unflattened.parquet")

In [4]:
df.head(5)

,GameId,PlayId,Team,X,Y,S,A,Dis,Orientation,Dir,...,Y_rel,Dir_sin,Dir_cos,Ori_sin,Ori_cos,Sx,Sy,IsRusher,IsOffense,Yards
0,2017090700,20170907000118,0.0,46.09,18.46,4.0,1.13,0.40,171.99,357.18,...,-8.19,0.998789,-0.049198,-0.990244,0.139346,-0.196794,3.995156,0,0,8
1,2017090700,20170907000118,0.0,45.33,20.66,0.1,1.35,0.01,117.61,18.70,...,-5.99,0.947210,0.320613,-0.463451,0.886123,0.032061,0.094721,0,0,8
2,2017090700,20170907000118,0.0,46.00,20.10,3.1,0.59,0.31,93.01,22.73,...,-6.55,0.922336,0.386389,-0.052510,0.998620,1.197806,2.859241,0,0,8
3,2017090700,20170907000118,0.0,48.54,25.60,0.2,0.54,0.02,89.77,285.64,...,-1.05,0.269592,-0.962975,0.004014,0.999992,-0.192595,0.053918,0,0,8
4,2017090700,20170907000118,0.0,50.68,17.88,1.6,2.43,0.16,102.63,344.31,...,-8.77,0.962739,-0.270432,-0.218654,0.975802,-0.432692,1.540382,0,0,8


In [5]:
print(df.shape)
print("\nLines by PlayID")
print(df.groupby("PlayId").size().value_counts().head())

(682154, 43)

Lines by PlayID
22    31007
Name: count, dtype: int64


Now, we will focus on the most important task for the baseline: feature engineering. We will do so in three steps. 

##Geometry of Pairwise Interactions

First step refers to geometry of pairwise interactions.

we already have relative coordinates (relative to LOS), vectorial components of angles, vectorial velocity.

Now, we will focus on Rusher ↔ Defender geometry
For each of the 11 defenders, we'll compute relative to the rusher:

A) Distances
def{i}_dist_to_rusher: Euclidean distance 
def{i}_dx_to_rusher: X-axis difference (ahead / behind)
def{i}_dy_to_rusher: Y-axis difference (lateral)

B) Angles

def{i}_angle_to_rusher_sin, def{i}_angle_to_rusher_cos: angle of the rusher→defender line (decomposed into sine/cosine)

C) Relative Kinematics

def{i}_rel_Sx, def{i}_rel_Sy: defender's velocity relative to the rusher (def_Sx - rusher_Sx)
def{i}_rel_speed: magnitude of the relative velocity
def{i}_closing_speed: approach velocity — projection of the relative velocity onto the vector connecting defender→rusher. Positive = approaching, negative = moving away. 

D) Approach Angle

def{i}_approach_angle_sin, def{i}_approach_angle_cos: angle between the defender's velocity vector and the defender→rusher vector. 0° = attacking directly, 90° = passing sideways.

E) Time-to-Collision (TTC)

def{i}_ttc: estimated time for the defender to reach the rusher, assuming straight (closest) trajectories. Calculated as the time that minimizes the distance between the two moving objects.

def{i}_min_dist: minimum projected distance on the future trajectory — if they never cross, it indicates how close they will pass.

Together with that, rusher ↔ Offensive Blocker interactions will be stored.
Analogous to the main but for the 10 non-rusher offensive players. The intuition is: nearby blockers moving in front of the rusher clear a path.For each of the 10 non-rusher offensive players:

off{i}_dist_to_rusher
off{i}_dx_to_rusher, off{i}_dy_to_rusher
off{i}_rel_speed, off{i}_closing_speed (interpreted as "going along" if negative)

In [6]:
def add_rusher_columns(df):
#Adds columns rusher_X_rel, rusher_Y_rel, rusher_Sx, rusher_Sy to aa rows (replicating the rusher value for that play)
    df = df.copy()
    rusher_rows = df[df["IsRusher"] == 1][
        ["PlayId", "X_rel", "Y_rel", "Sx", "Sy", "S"]].rename(columns={
        "X_rel": "rusher_X_rel",
        "Y_rel": "rusher_Y_rel",
        "Sx":    "rusher_Sx",
        "Sy":    "rusher_Sy",
        "S":     "rusher_S",})
    df = df.merge(rusher_rows, on="PlayId", how="left")
    return df


df = add_rusher_columns(df)
print(df.shape)

(682154, 48)


In [7]:
#A)Distances
df["dx_to_rusher"] = df["X_rel"] - df["rusher_X_rel"]
df["dy_to_rusher"] = df["Y_rel"] - df["rusher_Y_rel"]
df["dist_to_rusher"] = np.sqrt(df["dx_to_rusher"]**2 + df["dy_to_rusher"]**2)
df["dist_manhattan"] = np.abs(df["dx_to_rusher"]) + np.abs(df["dy_to_rusher"])

In [8]:
#B)Angles
safe_dist = df["dist_to_rusher"].replace(0, np.nan)
df["angle_to_rusher_cos"] = df["dx_to_rusher"] / safe_dist
df["angle_to_rusher_sin"] = df["dy_to_rusher"] / safe_dist

In [9]:
#C)Relative Kinematics
df["rel_Sx"] = df["Sx"] - df["rusher_Sx"]
df["rel_Sy"] = df["Sy"] - df["rusher_Sy"]
df["rel_speed"] = np.sqrt(df["rel_Sx"]**2 + df["rel_Sy"]**2)

#closing speed = projection of relative velocity onto the vector (rusher-player)
#sign: positive = approaching (closing), negative = moving away
#unit vector (player → rusher) = -1 * (player-relative-to-rusher vector) / dist
#closing speed = -(rel_velocity . unit_vector_player_to_rusher), equivalent to: (rusher_velocity - player_velocity) projected onto (rusher - player)

dot_product = (df["rel_Sx"] * df["dx_to_rusher"] +
               df["rel_Sy"] * df["dy_to_rusher"])
df["closing_speed"] = -dot_product / safe_dist


In [10]:
#D)Approach Angle
#0° = moving directly toward the rusher; 180° = moving in the opposite direction
safe_S = df["S"].replace(0, np.nan)
unit_to_rusher_x = -df["dx_to_rusher"] / safe_dist
unit_to_rusher_y = -df["dy_to_rusher"] / safe_dist
#player velocity vector (unit vector)
unit_v_x = df["Sx"] / safe_S
unit_v_y = df["Sy"] / safe_S
#dot product between the two unit vectors = cos(angle)
df["approach_angle_cos"] = (unit_v_x * unit_to_rusher_x +
                            unit_v_y * unit_to_rusher_y)
# sin = 2D cross product
df["approach_angle_sin"] = (unit_v_x * unit_to_rusher_y -
                            unit_v_y * unit_to_rusher_x)

In [11]:
#E)Time-to-Collision (TTC)
# t* = -(p0 . v_rel) / |v_rel|^2, p0 = player - rusher
rel_speed_sq = df["rel_speed"]**2
safe_rel_speed_sq = rel_speed_sq.replace(0, np.nan)
t_star = -(df["dx_to_rusher"] * df["rel_Sx"] + df["dy_to_rusher"] * df["rel_Sy"]) / safe_rel_speed_sq

#TTC: if t_star < 0, players are moving apart (assign high value)
#if rel_speed ~ 0, velocities are parallel (assign high value)
df["ttc"] = np.where(t_star > 0, t_star, 999.0)
df["ttc"] = np.where(df["rel_speed"] < 0.1, 999.0, df["ttc"])
#projected minimum distance (how close will they get?)
proj = (df["dx_to_rusher"] * df["rel_Sx"] +
        df["dy_to_rusher"] * df["rel_Sy"])
d_min_sq = df["dist_to_rusher"]**2 - (proj**2 / safe_rel_speed_sq)
df["min_dist_projected"] = np.sqrt(np.clip(d_min_sq, 0, None))
df["min_dist_projected"] = df["min_dist_projected"].fillna(df["dist_to_rusher"])

In [12]:
#distribution of the new features
kin_features = ["dist_to_rusher", "dx_to_rusher", "dy_to_rusher",
    "rel_speed", "closing_speed",
    "approach_angle_cos", "approach_angle_sin",
    "ttc", "min_dist_projected"]

print("\nkinemactic features' distributions")
print(df[df["IsOffense"] == 0][kin_features].describe().round(2))


kinemactic features' distributions
       dist_to_rusher  dx_to_rusher  dy_to_rusher  rel_speed  closing_speed  \
count       341077.00     341077.00     341077.00  341050.00      341050.00   
mean            10.35          8.01          0.01       4.50           2.90   
std              5.87          4.81          7.37       1.68           2.62   
min              0.23        -12.41        -34.31       0.00         -12.08   
25%              5.90          4.77         -3.88       3.36           1.65   
50%              8.03          6.62         -0.04       4.42           3.30   
75%             14.13          9.41          3.80       5.54           4.60   
max             55.41         54.38         39.04      18.98          18.69   

       approach_angle_cos  approach_angle_sin        ttc  min_dist_projected  
count           339823.00           339823.00  341077.00           341077.00  
mean                 0.21                0.01     126.48                5.56  
std            

Also, we can analyze Nearest defender ↔ blockers interactions.
The idea is to capture individual matchups: for each defender, what is the closest offensive blocker to them? For each defender:

def{i}_nearest_blocker_dist: distance to the nearest offensive blocker

def{i}_nearest_blocker_dx, dy: coordinate differences

def{i}_is_engaged: binary flag — if within ≤1 yard of the blocker, it is considered "engaged"

In [13]:
#new features
df["nearest_blocker_dist"] = np.nan
df["nearest_blocker_dx"] = np.nan
df["nearest_blocker_dy"] = np.nan
df["is_engaged"] = np.nan
#just to add everything to the df at once at the end
all_updates = []  
for play_id, group in df.groupby("PlayId", sort=False):
    defenders = group[group["IsOffense"] == 0]
    blockers = group[(group["IsOffense"] == 1)&(group["IsRusher"] == 0)]
    if len(defenders) == 0 or len(blockers) == 0:
        continue

def_xy = defenders[["X_rel", "Y_rel"]].values#(11, 2)
block_xy = blockers[["X_rel", "Y_rel"]].values#(10, 2) (exc rusher)
#dist mat
diffs = def_xy[:, np.newaxis, :] - block_xy[np.newaxis, :, :]  #(11, 10, 2)
dists = np.sqrt((diffs ** 2).sum(axis=2)) #(11, 10)
#pairing up
nearest_idx = dists.argmin(axis=1) #(11,)
nearest_dist = dists[np.arange(len(defenders)), nearest_idx] #(11,)
nearest_dx = diffs[np.arange(len(defenders)), nearest_idx, 0]
nearest_dy = diffs[np.arange(len(defenders)), nearest_idx, 1]
engaged = (nearest_dist <= 1.0).astype(float)

for idx, dist, dx, dy, eng in zip(defenders.index, nearest_dist, nearest_dx, nearest_dy, engaged):
    all_updates.append((idx, dist, dx, dy, eng))
#apply all
if all_updates:
    idxs = [u[0] for u in all_updates]
    dists = [u[1] for u in all_updates]
    dxs = [u[2] for u in all_updates]
    dys = [u[3] for u in all_updates]
    engs = [u[4] for u in all_updates]
    df.loc[idxs, "nearest_blocker_dist"] = dists
    df.loc[idxs, "nearest_blocker_dx"] = dxs
    df.loc[idxs, "nearest_blocker_dy"] = dys
    df.loc[idxs, "is_engaged"] = engs

In [14]:
#for blocker only: are they in front of the rusher?
df["is_lead_blocker"] = ((df["IsOffense"] == 1)&(df["IsRusher"] == 0)&(df["dx_to_rusher"] > 0)).astype(int)
lead_count = df[df["is_lead_blocker"] == 1].groupby("PlayId").size()

##Density and Space Usage

Now, we can evaluate, for each play:

A) def_within_{r}yd

how many defenders are within each radius of the rusher? We already calculated this in EDA, but now it will be included as a feature

B) dist_forward,dx_forward, dy_forward 

what's in between the rb and the end zone?

C) gap_

given that gaps = blank spaces in Oline. The idea is to divide the Y-axis into 2-3 yard strips around the rusher and count defenders in each

D) void_

out of all gaps, which's the most fruitfull Yards-wise

In [17]:
radii = [2, 5, 7]
play_features = []
for play_id, group in df.groupby("PlayId", sort=False):
    defenders = group[group["IsOffense"] == 0]
    row = {"PlayId": play_id}
    for r in radii:
        row[f"def_within_{r}yd"]=(defenders["dist_to_rusher"]<=r).sum()
    play_features.append(row)

Now, let's capture what's in between the rb and the end zone

In [18]:
for play_id, group in df.groupby("PlayId", sort=False):
    defenders = group[group["IsOffense"] == 0]
    #in front since EZ's to the front
    forward_def = defenders[defenders["dx_to_rusher"] > 0]
    row = {"PlayId": play_id}
    row["n_def_forward"] = len(forward_def)
    #min and avg distces
    if len(forward_def) > 0:
        row["mean_dist_forward_def"]=forward_def["dist_to_rusher"].mean()
        row["min_dist_forward_def"]=forward_def["dist_to_rusher"].min()
        #x coomponent only
        row["mean_dx_forward"]=forward_def["dx_to_rusher"].mean()
        #sidelines dispersion
        row["std_dy_forward"]=forward_def["dy_to_rusher"].std()
    else:
        row["mean_dist_forward_def"] = 99.0
        row["min_dist_forward_def"] = 99.0
        row["mean_dx_forward"] = 99.0
        row["std_dy_forward"] = 0.0
    
    row["closest_def_dist"] = defenders["dist_to_rusher"].min()
    #which combines proximity + speed of approach
    eps = 0.5
    threat_score = defenders["closing_speed"] / (defenders["dist_to_rusher"] + eps)
    row["max_threat_score"] = threat_score.max()
    play_features.append(row)

Gaps = blank spaces in Oline. The idea is to divide the Y-axis into 2-3 yard strips around the rusher and count defenders in each

In [19]:
for play_id, group in df.groupby("PlayId", sort=False):
    defenders = group[group["IsOffense"] == 0]
    relevant = defenders[(defenders["dx_to_rusher"] > 0)&(defenders["dx_to_rusher"] <= 10)]
    row = {"PlayId": play_id}
    row["gap_center"] = ((relevant["dy_to_rusher"].abs() < 1)).sum()
    row["gap_inside_left"] = ((relevant["dy_to_rusher"] >= -3)&(relevant["dy_to_rusher"] < -1)).sum()
    row["gap_inside_right"] = ((relevant["dy_to_rusher"] >  1)&(relevant["dy_to_rusher"] <= 3)).sum()
    row["gap_outside_left"]= ((relevant["dy_to_rusher"] >= -6)&(relevant["dy_to_rusher"] < -3)).sum()
    row["gap_outside_right"] = ((relevant["dy_to_rusher"] >  3) & (relevant["dy_to_rusher"] <= 6)).sum()
    play_features.append(row)

Out of all gaps, which's the most fruitfull Yards-wise

In [20]:
dx_grid = np.linspace(1, 10, 10)
dy_grid = np.linspace(-10, 10, 21)
DX, DY = np.meshgrid(dx_grid, dy_grid)
candidate_points = np.column_stack([DX.ravel(), DY.ravel()])#(210, 2)

for play_id, group in df.groupby("PlayId", sort=False):
    defenders = group[group["IsOffense"] == 0]
    def_pos = defenders[["dx_to_rusher", "dy_to_rusher"]].values #(11, 2)
    #candidate points
    diffs = candidate_points[:, np.newaxis, :]-def_pos[np.newaxis, :, :]
    dists = np.sqrt((diffs ** 2).sum(axis=2))  #(210, 11)
    min_dists = dists.min(axis=1)#(210,)
    #most fruitful=stronger deffensive freedom
    best_idx = min_dists.argmax()
    void_dx, void_dy = candidate_points[best_idx]#best space (in front and sideways)
    void_freedom = min_dists[best_idx] #like a freedom magnitude
    #on avg
    avg_freedom = min_dists.mean()

    row = {"PlayId": play_id,"void_dx": void_dx, "void_dy": void_dy,
    "void_freedom": void_freedom,"avg_freedom_forward": avg_freedom, }
    play_features.append(row)

In [23]:
play_features = pd.DataFrame(play_features)
print(play_features.describe().round(2))

             PlayId  def_within_2yd  def_within_5yd  def_within_7yd  \
count  1.240280e+05        31007.00        31007.00        31007.00   
mean   2.017975e+13            0.01            1.34            4.31   
std    7.844619e+09            0.09            1.28            1.52   
min    2.017091e+13            0.00            0.00            0.00   
25%    2.017113e+13            0.00            0.00            3.00   
50%    2.018101e+13            0.00            1.00            4.00   
75%    2.019091e+13            0.00            2.00            5.00   
max    2.019113e+13            5.00           11.00           11.00   

       n_def_forward  mean_dist_forward_def  min_dist_forward_def  \
count       31007.00               31007.00              31007.00   
mean           10.99                  10.36                  4.58   
std             0.10                   1.49                  0.97   
min             8.00                   2.39                  0.23   
25%            

In [1]:
#df = df.merge(play_features, on="PlayId", how="left")

##Contextual Features 

Now, we will focus on more subtle/nuanced features.

A) Momentum 

The idea here is that a 90kg cornerback running at 7m/s is a different threat than a 110kg lineback at the same speed, since the impact on the blocker is going to be another. We can evaluate the vectorial components and the magnitude of momentum.
Even though our data here is given in pounds and yards/second, we wont convert anything to ISM. The models we plan on evaluating are tree based, and those are not affected by scale. The only import point here is adopting a single unit throughout the whole dataset. 

B) Scoreboard

Evidently, a team's strategy changes depending on if it's winning or not, just as it changes as time approaches the end (as as shown previously). So with this feature we aim on capturing this advantage or disadvantage from attacker's pov.

C) RB's control area 

A RB with a large cell has open space ahead; with a small cell, it is trapped. We opted for a Voronoi tessellation based because Voronoi divides the field into "dominance" regions (each point on the field belongs to the nearest player). The rb's cell is the set of points on the field where he is the nearest player, which means this space is immediately available to him before any defender arrives.
This captures exactly the question that matters to the rusher: "where can I go before someone reaches me?". Voronoi answers this as it measures the single agent dominance against a set of deffensemen.

D) Deffense's hull area and Rb's distance to it

Differently from the RB, defense does not act as isolated individuals, but as a collective structure. The convex hull measures the geometric dominance of the defense (how large/spread/compact it is, and crucially, where the rusher is in relation to that map). The tactical question it answers is: "is the rusher surrounded, about to break through, or has he already broken through?". If he is well inside the defensive hull, there is still a wall. If it's near the edge, it's about to break out. If it's outside, it's already broken through the defense and an explosive run is almost certain.

E) RB speed in void direction

Directly, only the vectorial velocity it has on the direction it can effectively make progress. 

In [2]:
#just some checks before picking back up since i did this through several days
# #Which columns are missing from the situational function?
# needed = [
#     "PossessionTeam", "HomeTeamAbbr", "VisitorTeamAbbr",
#     "HomeScoreBeforePlay", "VisitorScoreBeforePlay",
#     "Quarter", "Down", "Distance", "GameClock",
#     "AbsYardLine",]

# print("Status:")
# for col in needed:
#     status = "Yes" if col in df.columns else "No"
#     print(f"  {col}: {status}")

In [3]:
# missing_cols = [c for c in needed if c not in df.columns]
# print(f"raw: {missing_cols}")

# if missing_cols:
#     cols_to_load = ["PlayId"] + [c for c in missing_cols if c != "AbsYardLine"]
#     df_raw_extra = pd.read_csv("train.csv", low_memory=False, usecols=cols_to_load)
#     df_raw_extra = df_raw_extra.drop_duplicates(subset="PlayId")
#     df = df.merge(df_raw_extra, on="PlayId", how="left")

# for col in needed:
#     status = "✓" if col in df.columns else "✗ AINDA AUSENTE"
#     print(f"  {col}: {status}")

In [18]:
df["IsHomeOffense"] = (df["PossessionTeam"] == df["HomeTeamAbbr"]).astype(int)
df["ScoreDiff"] = np.where(df["IsHomeOffense"] == 1,df["HomeScoreBeforePlay"] - df["VisitorScoreBeforePlay"],df["VisitorScoreBeforePlay"]-df["HomeScoreBeforePlay"])
#Goal-to-go: the distance to the first down. It's the distance to the end zone (within 10 yards or less)
df["is_goal_to_go"] = ((df["AbsYardLine"] >= 90) & (df["Distance"] <= (110 - df["AbsYardLine"]))).astype(int)
#two-minute drill: urgency on 2 or 4 quarter
df["is_two_minute_drill"] = ((df["Quarter"].isin([2, 4])) &(df["GameClockPct"] > 0.867)).astype(int)
#scoreboard X time
#team winning big in Q4, defensively killing the clock. Defense knows this and attacks, yards drop
df["is_winning_late"] = ((df["Quarter"] == 4) & (df["ScoreDiff"] >= 7)).astype(int)
##team losing a lot in Q4, rarely race, catches defense by surprise
df["is_trailing_late"] = ((df["Quarter"] == 4) & (df["ScoreDiff"] <= -7)).astype(int)

In [26]:
FIELD_BBOX = Polygon([(-30, -30), (30, -30), (30, 30), (-30, 30)])
spatial_features = []
for play_id, group in df.groupby("PlayId", sort=False):
    rusher = group[group["IsRusher"] == 1].iloc[0]
    defenders = group[group["IsOffense"] == 0]
    all_players = group[["X_rel", "Y_rel"]].values  # (22, 2)
    rusher_xy = np.array([rusher["X_rel"], rusher["Y_rel"]])
    rusher_idx = group.reset_index(drop=True).index[group["IsRusher"] == 1][0]
    row = {"PlayId": play_id}
    try:
        ghost = np.array([[-1000, 0], [1000, 0], [0, -1000], [0, 1000]])
        points_with_ghosts = np.vstack([all_players, ghost])
        vor = Voronoi(points_with_ghosts)
        region_idx = vor.point_region[rusher_idx]
        region = vor.regions[region_idx]
        if -1 in region or len(region) == 0:
            row["rusher_voronoi_area"] = np.nan
        else:
            vertices = vor.vertices[region]
            poly = Polygon(vertices)
            #bound clipping
            poly_clipped = poly.intersection(FIELD_BBOX)
            row["rusher_voronoi_area"] = poly_clipped.area
    except Exception:
        row["rusher_voronoi_area"] = np.nan

    def_xy = defenders[["X_rel", "Y_rel"]].values
    try:
        hull_full = ConvexHull(def_xy)
        row["defense_hull_area"] = hull_full.volume  
        hull_polygon = Polygon(def_xy[hull_full.vertices])
        rusher_point = Point(rusher_xy)
        unsigned_dist = hull_polygon.boundary.distance(rusher_point)
        if hull_polygon.contains(rusher_point):
            row["dist_to_hull_boundary"] = -unsigned_dist   
        else:
            row["dist_to_hull_boundary"] = unsigned_dist    
    except Exception:
        row["defense_hull_area"] = np.nan
        row["dist_to_hull_boundary"] = np.nan
    #up to 10y from LOS
    box_defenders = defenders[defenders["X_rel"].between(-2, 10)]
    if len(box_defenders) >= 3:
        try:
            box_xy = box_defenders[["X_rel", "Y_rel"]].values
            hull_box = ConvexHull(box_xy)
            row["def_hull_area_box"] = hull_box.volume
        except Exception:
            row["def_hull_area_box"] = np.nan
    else:
        row["def_hull_area_box"] = 0.0   
    void_dx = rusher["void_dx"]
    void_dy = rusher["void_dy"]
    void_norm = np.sqrt(void_dx**2 + void_dy**2)
    if void_norm > 0:
        void_unit_x = void_dx / void_norm
        void_unit_y = void_dy / void_norm
        row["rusher_speed_in_void_direction"] = (rusher["Sx"] * void_unit_x + rusher["Sy"] * void_unit_y)
    else:
        row["rusher_speed_in_void_direction"] = 0.0

    spatial_features.append(row)

spatial_features = pd.DataFrame(spatial_features)

In [29]:
df = df.merge(spatial_features, on="PlayId", how="left")

In [30]:
df.to_parquet("df_training_finalidk.parquet", index=False)

In [ ]:
#now, we will flatten the dataset

In [ ]:
#organizes order of players by distance from rb to flatten dataset (position invariance thing)
# PLAYER_FEATURES_BASE = ["X_rel", "Y_rel", "S", "A", "Dis",
#     "Dir_sin", "Dir_cos", "Ori_sin", "Ori_cos", "Sx", "Sy",
#     "PlayerHeight", "PlayerWeight", "PlayerAge","Position",]

# PAIRWISE_FEATURES = ["dist_to_rusher", "dx_to_rusher", "dy_to_rusher",
#     "rel_Sx", "rel_Sy", "rel_speed", "closing_speed","approach_angle_cos", "approach_angle_sin",
#     "ttc", "min_dist_projected",]

# PLAY_FEATURES = ["GameId","Season", "Week", "Quarter", "Down", "Distance",
#     "HomeScoreBeforePlay", "VisitorScoreBeforePlay", "OffenseFormation", "OffensePersonnel", 
#     "DefensePersonnel", "DefendersInTheBox", "PossessionTeam", "FieldPosition",
#     "AbsYardLine","GameClockPct",]

# def flatten_play_ordered(group):
# #Flattens a play (22 lines) into 1 line wide. [rusher, offensive (ordered by distance to the rusher, ascending), defensive (ordered by distance to the rusher, ascending)]

#     rusher  = group[group["IsRusher"] == 1]
#     offense = group[(group["IsRusher"] == 0) & (group["IsOffense"] == 1)]
#     defense = group[group["IsOffense"] == 0]
#     offense = offense.sort_values("dist_to_rusher").reset_index(drop=True)
#     defense = defense.sort_values("dist_to_rusher").reset_index(drop=True)

#     row = {}

#     #play feats
#     for f in PLAY_FEATURES:
#         if f in group.columns:
#             row[f] = group.iloc[0][f]
#     #rusher
#     rusher_row = rusher.iloc[0]
#     for f in PLAYER_FEATURES_BASE:
#         row[f"rusher_{f}"] = rusher_row[f]
#     #offensive
#     for i in range(10):
#         if i < len(offense):
#             player = offense.iloc[i]
#             for f in PLAYER_FEATURES_BASE:
#                 row[f"off{i+1}_{f}"] = player[f]
#         else:
#             for f in PLAYER_FEATURES_BASE:
#                 row[f"off{i+1}_{f}"] = np.nan
#     #deffensive
#     for i in range(11):
#         if i < len(defense):
#             player = defense.iloc[i]
#             for f in PLAYER_FEATURES_BASE:
#                 row[f"def{i+1}_{f}"] = player[f]
#             for f in PAIRWISE_FEATURES:
#                 row[f"def{i+1}_{f}"] = player[f]
#         else:
#             for f in PLAYER_FEATURES_BASE + PAIRWISE_FEATURES:
#                 row[f"def{i+1}_{f}"] = np.nan

#     return pd.Series(row)

In [3]:
df = pd.read_parquet("df_training_finalidk.parquet")

In [4]:
df.head(5)

,GameId,PlayId,Team,X,Y,S,A,Dis,Orientation,Dir,...,Quarter,GameClock,is_two_minute_drill,is_winning_late,is_trailing_late,rusher_voronoi_area,defense_hull_area,dist_to_hull_boundary,def_hull_area_box,rusher_speed_in_void_direction
0,2017090700,20170907000118,0.0,46.09,18.46,4.0,1.13,0.40,171.99,357.18,...,1,14:14:00,0,0,0,0.0,207.8742,3.980704,0.0,0.0
1,2017090700,20170907000118,0.0,46.09,18.46,4.0,1.13,0.40,171.99,357.18,...,1,14:14:00,0,0,0,0.0,207.8742,3.980704,0.0,0.0
2,2017090700,20170907000118,0.0,46.09,18.46,4.0,1.13,0.40,171.99,357.18,...,1,14:14:00,0,0,0,0.0,207.8742,3.980704,0.0,0.0
3,2017090700,20170907000118,0.0,46.09,18.46,4.0,1.13,0.40,171.99,357.18,...,1,14:14:00,0,0,0,0.0,207.8742,3.980704,0.0,0.0
4,2017090700,20170907000118,0.0,45.33,20.66,0.1,1.35,0.01,117.61,18.70,...,1,14:14:00,0,0,0,0.0,207.8742,3.980704,0.0,0.0


In [5]:
print(list(df.columns))

['GameId', 'PlayId', 'Team', 'X', 'Y', 'S', 'A', 'Dis', 'Orientation', 'Dir', 'NflId', 'Season', 'YardLine', 'PossessionTeam', 'Down', 'Distance', 'FieldPosition', 'HomeScoreBeforePlay', 'VisitorScoreBeforePlay', 'NflIdRusher', 'OffenseFormation', 'OffensePersonnel', 'DefendersInTheBox', 'DefensePersonnel', 'PlayerHeight', 'PlayerWeight', 'Position', 'Week', 'PlayerAge', 'GameClockPct', 'GameElapsedPct', 'AbsYardLine', 'X_rel', 'Y_rel', 'Dir_sin', 'Dir_cos', 'Ori_sin', 'Ori_cos', 'Sx', 'Sy', 'IsRusher', 'IsOffense', 'Yards', 'rusher_X_rel', 'rusher_Y_rel', 'rusher_Sx', 'rusher_Sy', 'rusher_S', 'dx_to_rusher', 'dy_to_rusher', 'dist_to_rusher', 'dist_manhattan', 'angle_to_rusher_cos', 'angle_to_rusher_sin', 'rel_Sx', 'rel_Sy', 'rel_speed', 'closing_speed', 'approach_angle_cos', 'approach_angle_sin', 'ttc', 'min_dist_projected', 'nearest_blocker_dist', 'nearest_blocker_dx', 'nearest_blocker_dy', 'is_engaged', 'is_lead_blocker', 'def_within_2yd', 'def_within_5yd', 'def_within_7yd', 'n_def_

In [6]:
def process_chunks(df_long, chunk_size=500, output_filename="nfl.parquet"):
    #to not split games across chunks
    unique_plays = df_long['PlayId'].unique()
    total_plays = len(unique_plays)
    #downcast to reduce RAM
    for col in df_long.columns:
        if df_long[col].dtype == 'float64':
            df_long[col] = df_long[col].astype('float32')
        elif df_long[col].dtype == 'int64':
            df_long[col] = df_long[col].astype('int32')

    is_first_chunk = True
    #through PlayIds
    for i in range(0, total_plays, chunk_size):
        chunk_ids = unique_plays[i : i + chunk_size]
        df_chunk_long = df_long[df_long['PlayId'].isin(chunk_ids)].copy()
        df_chunk_flat = flatten_chunk(df_chunk_long)
        if is_first_chunk:
            df_chunk_flat.to_csv(output_filename, index=False, mode='w')
            is_first_chunk = False
        else:
            df_chunk_flat.to_csv(output_filename, index=False, mode='a', header=False)

        #free memory
        del df_chunk_long, df_chunk_flat
        gc.collect()

In [7]:
def flatten_chunk(df_chunk):
    #find rb via 'IsRusher'
    df_rusher = df_chunk[df_chunk['IsRusher'] == 1].copy()
    #context features
    context_columns = ['GameId', 'PlayId', 'Season', 'YardLine', 'PossessionTeam', 'Down', 'Distance','FieldPosition', 'HomeScoreBeforePlay', 'VisitorScoreBeforePlay', 'OffenseFormation','DefendersInTheBox', 'Quarter', 'GameClock', 'is_two_minute_drill', 'is_winning_late','is_trailing_late', 'defense_hull_area', 'def_hull_area_box', 'Yards']
    valid_context_columns = [col for col in context_columns if col in df_chunk.columns]
    df_context = (df_chunk[valid_context_columns].drop_duplicates(subset=['PlayId']).copy())
    df_players = df_chunk[df_chunk['IsRusher'] != 1].copy()
    #player features that will be expanded horizontally
    player_features = ['X_rel', 'Y_rel', 'Sx', 'Sy', 'S', 'A', 'Dis','Ori_cos', 'Ori_sin', 'Dir_cos', 'Dir_sin','dist_to_rusher', 'ttc', 'dist_to_hull_boundary','closest_def_dist', 'max_threat_score']
    player_features = [col for col in player_features if col in df_players.columns]
    #sort offense players by distance to the running back
    df_offense = df_players[df_players['IsOffense'] == 1].copy()
    df_offense = df_offense.sort_values(by=['PlayId', 'dist_to_rusher'])
    df_offense['player_rank'] = (df_offense.groupby('PlayId').cumcount() + 1).astype('int8')
    wide_offense = df_offense.pivot(index='PlayId',columns='player_rank',values=player_features)
    wide_offense.columns = [f'offense_{col[0]}_{col[1]}'for col in wide_offense.columns]
    wide_offense = wide_offense.reset_index()
    del df_offense
    #sort defense players by distance to the running back
    df_defense = df_players[df_players['IsOffense'] == 0].copy()
    df_defense = df_defense.sort_values(by=['PlayId', 'dist_to_rusher'])
    df_defense['player_rank'] = (df_defense.groupby('PlayId').cumcount() + 1
).astype('int8')
    wide_defense = df_defense.pivot(index='PlayId',columns='player_rank',values=player_features)
    wide_defense.columns = [f'defense_{col[0]}_{col[1]}'for col in wide_defense.columns]
    wide_defense = wide_defense.reset_index()
    del df_defense, df_players
    #running back features
    rusher_columns = ['GameId', 'PlayId', 'X_rel', 'Y_rel', 'Sx', 'Sy','S', 'A', 'rusher_voronoi_area','rusher_speed_in_void_direction', 'void_freedom']
    valid_rusher_columns = [col for col in rusher_columns if col in df_rusher.columns]
    df_rusher_features = df_rusher[valid_rusher_columns].copy()
    df_rusher_features.columns = [f'rusher_{col}' if col not in ['GameId', 'PlayId'] else col for col in df_rusher_features.columns]
    del df_rusher
    #merge all data together
    df_flat = df_context.merge(df_rusher_features,on='PlayId',how='inner')
    del df_context, df_rusher_features
    df_flat = df_flat.merge(wide_offense,on='PlayId',how='inner')
    del wide_offense
    df_flat = df_flat.merge(wide_defense,on='PlayId',how='inner')
    del wide_defense

    return df_flat.fillna(0)

In [8]:
process_chunks(df)

In [ ]:
#finally, we will merge the wide dataset's target columns with the flattened version 

In [9]:
df_wide = pd.read_parquet("df_training.parquet")
for col in df_wide.columns:
    if df_wide[col].dtype == 'float64':
        df_wide[col] = df_wide[col].astype('float32')
    elif df_wide[col].dtype == 'int64':
        df_wide[col] = df_wide[col].astype('int32')

if 'Yards' not in df_wide.columns:
    df_target = df_large[['GameId', 'PlayId', 'Yards']].drop_duplicates(subset=['PlayId'])
    df_wide = df_wide.merge(df_target, on=['GameId', 'PlayId'], how='inner')
    del df_target
    gc.collect()

df_wide.to_parquet("nfl.parquet", index=False, compression="snappy")
del df_wide
gc.collect()

0